# 01 · Data preparation and midfielder screening

**Main workflow. No SQL Server required.** Input: cached/downloaded IMPECT season JSON. Output: core_analysis.pkl plus candidate CSVs and raw-file validation evidence. This is the independently computed Python reference for the SQL model. Raw JSON is around 1.1 GB; loading and flattening the complete season takes time and memory.

Run cells from top to bottom in a **fresh kernel**. The setup cell declares inputs explicitly. Code comments and explanations are in English. The original source cell numbers are recorded in notebook metadata and in `docs/cell_mapping.csv`.

## Setup

Locate the repository and import shared utilities. This does not change the database.

In [1]:
from pathlib import Path
import sys
import json
from urllib.request import urlopen
import numpy as np
import pandas as pd
from IPython.display import display

# Find the repository from any notebook subfolder.
root = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "src" / "scouting" / "config.py").is_file()), None)
if root is None:
    raise FileNotFoundError("Open this notebook inside the extracted repository.")
if str(root / "src") not in sys.path:
    sys.path.insert(0, str(root / "src"))
from scouting.config import PROJECT_ROOT, RAW_DIR, PROCESSED_DIR, REPORTS_DIR, IMPECT_BASE_URL, ITERATION_ID
from scouting.data_access import load_or_download_json, load_metadata_context, metadata
from scouting.checkpoints import save_context, load_context, BASE_VARIABLES, ADVANCED_VARIABLES
print("Project:", PROJECT_ROOT)

globals().update(load_metadata_context())

Project: /workspace/scratch/491a21a4af19/organized_project/football-no8-recruitment


## Set portable project paths

Use the repository root found by the setup cell. Data is cached under data/raw/impect; no personal Windows path is required.

In [2]:
# Paths are provided by the shared setup cell.
for folder in ["events", "player_kpis"]:
    (RAW_DIR / folder).mkdir(parents=True, exist_ok=True)
print("Data destination:", RAW_DIR.resolve())


Data destination: /workspace/scratch/491a21a4af19/audit_input/data/raw/impect


## Test the cache on three matches

Read cached JSON when it exists and download only missing files. A small pilot catches path or source problems early.

In [3]:
pilot_match_ids = (
    available_matches
    .sort_values(["scheduledDate", "id"])
    ["id"]
    .head(3)
    .astype(int)
    .tolist()
)

download_log = []

for match_id in pilot_match_ids:
    for dataset in ["events", "player_kpis"]:
        filename = f"{dataset}_{match_id}.json"

        url = f"{IMPECT_BASE_URL}/{dataset}/{filename}"
        destination = RAW_DIR / dataset / filename

        try:
            data, status = load_or_download_json(url, destination)

            download_log.append({
                "match_id": match_id,
                "dataset": dataset,
                "status": status,
                "error": "",
            })

        except Exception as error:
            download_log.append({
                "match_id": match_id,
                "dataset": dataset,
                "status": "FAILED",
                "error": str(error),
            })

download_report = pd.DataFrame(download_log)
display(download_report)

   match_id      dataset  status error
0    122838       events  cached      
1    122838  player_kpis  cached      
2    122839       events  cached      
3    122839  player_kpis  cached      
4    122840       events  cached      
5    122840  player_kpis  cached      


## Cache the complete season

Check/download 306 event exports and 306 player-aggregate exports. The manifest records progress and failures so an interrupted download can resume.

In [4]:
import time
from datetime import datetime, timezone

# Unique match IDs in chronological order.
season_match_ids = (
    available_matches
    .sort_values(["scheduledDate", "id"])
    ["id"]
    .drop_duplicates()
    .astype(int)
    .tolist()
)

season_log = []
total_files = len(season_match_ids) * 2

for match_id in season_match_ids:
    for dataset in ["events", "player_kpis"]:
        filename = f"{dataset}_{match_id}.json"
        url = f"{IMPECT_BASE_URL}/{dataset}/{filename}"
        destination = RAW_DIR / dataset / filename

        status = "FAILED"
        error_message = ""

        # Try up to three times if a request or file read fails.
        for attempt in range(1, 4):
            try:
                _, status = load_or_download_json(url, destination)
                error_message = ""
                break

            except Exception as error:
                error_message = str(error)

                if attempt < 3:
                    time.sleep(attempt)

        season_log.append({
            "match_id": match_id,
            "dataset": dataset,
            "status": status,
            "attempts": attempt,
            "error": error_message,
            "checked_at_utc": datetime.now(
                timezone.utc
            ).isoformat(),
        })

        # Small pause after network requests.
        if status != "cached":
            time.sleep(0.2)

        completed = len(season_log)

        # Save progress and print an update every 20 files.
        if completed % 20 == 0 or completed == total_files:
            report = pd.DataFrame(season_log)
            report.to_csv(
                RAW_DIR / "download_manifest.csv",
                index=False,
            )

            failures = report["status"].eq("FAILED").sum()

            print(
                f"Processed {completed}/{total_files} files "
                f"| Failed: {failures}",
                flush=True,
            )

season_download_report = pd.DataFrame(season_log)

print("\nFINAL SUMMARY")
display(
    season_download_report.groupby(["dataset", "status"])
    .size()
    .reset_index(name="file_count")
)

failed_downloads = season_download_report.loc[
    season_download_report["status"].eq("FAILED")
]

if not failed_downloads.empty:
    display(failed_downloads)

Processed 20/612 files | Failed: 0
Processed 40/612 files | Failed: 0
Processed 60/612 files | Failed: 0
Processed 80/612 files | Failed: 0
Processed 100/612 files | Failed: 0
Processed 120/612 files | Failed: 0
Processed 140/612 files | Failed: 0
Processed 160/612 files | Failed: 0
Processed 180/612 files | Failed: 0
Processed 200/612 files | Failed: 0
Processed 220/612 files | Failed: 0
Processed 240/612 files | Failed: 0
Processed 260/612 files | Failed: 0
Processed 280/612 files | Failed: 0
Processed 300/612 files | Failed: 0
Processed 320/612 files | Failed: 0
Processed 340/612 files | Failed: 0
Processed 360/612 files | Failed: 0
Processed 380/612 files | Failed: 0
Processed 400/612 files | Failed: 0
Processed 420/612 files | Failed: 0
Processed 440/612 files | Failed: 0
Processed 460/612 files | Failed: 0
Processed 480/612 files | Failed: 0
Processed 500/612 files | Failed: 0
Processed 520/612 files | Failed: 0
Processed 540/612 files | Failed: 0
Processed 560/612 files | Failed

## Validate each raw file

Check structure, match IDs and team references before combining the season. These are structural checks, not proof of complete observation.

In [5]:
validation_rows = []

match_lookup = available_matches.set_index("id")

for match_id in season_match_ids:
    issues = []
    event_count = None
    player_record_count = None

    # 1. Check the event file.
    try:
        event_path = RAW_DIR / "events" / f"events_{match_id}.json"

        with event_path.open("r", encoding="utf-8") as file:
            match_events = json.load(file)

        if not isinstance(match_events, list) or not match_events:
            issues.append("Events must be a non-empty list")
        else:
            event_count = len(match_events)

            required_fields = {"id", "index", "actionType"}
            invalid_events = sum(
                not isinstance(event, dict)
                or not required_fields.issubset(event)
                for event in match_events
            )

            if invalid_events:
                issues.append(
                    f"{invalid_events} events missing required structure"
                )

    except Exception as error:
        issues.append(f"Event file: {error}")

    # 2. Check the player-statistics file.
    try:
        kpi_path = (
            RAW_DIR / "player_kpis" / f"player_kpis_{match_id}.json"
        )

        with kpi_path.open("r", encoding="utf-8") as file:
            match_kpis = json.load(file)

        if match_kpis["matchId"] != match_id:
            issues.append("Player-statistics match ID mismatch")

        expected_match = match_lookup.loc[match_id]

        if match_kpis["squadHome"]["id"] != expected_match["homeSquadId"]:
            issues.append("Home team mismatch")

        if match_kpis["squadAway"]["id"] != expected_match["awaySquadId"]:
            issues.append("Away team mismatch")

        home_players = match_kpis["squadHome"]["players"]
        away_players = match_kpis["squadAway"]["players"]

        if not isinstance(home_players, list) or not home_players:
            issues.append("Home player list missing or empty")

        if not isinstance(away_players, list) or not away_players:
            issues.append("Away player list missing or empty")

        player_record_count = len(home_players) + len(away_players)

    except Exception as error:
        issues.append(f"Player-statistics file: {error}")

    validation_rows.append({
        "match_id": match_id,
        "event_count": event_count,
        "player_position_records": player_record_count,
        "status": "PASS" if not issues else "REVIEW",
        "issues": " | ".join(issues),
    })

validation_report = pd.DataFrame(validation_rows)

print("Matches checked:", len(validation_report))
print(validation_report["status"].value_counts().to_string())

print("\nEvent counts per match:")
print(validation_report["event_count"].describe().round(0).to_string())

matches_to_review = validation_report.loc[
    validation_report["status"].eq("REVIEW")
]

if not matches_to_review.empty:
    display(matches_to_review)

Matches checked: 306
status
PASS    306

Event counts per match:
count     306.0
mean     3069.0
std       320.0
min      2253.0
25%      2842.0
50%      3078.0
75%      3297.0
max      4061.0


## Build the season event table

Keep the 16 documented provider fields plus match_id, preserve missing values, and verify row counts and duplicate event keys.

In [6]:
# Start only if every match passed the initial validation.
if not validation_report["status"].eq("PASS").all():
    raise ValueError("Resolve the validation issues first.")

event_columns = [
    "id",
    "index",
    "periodId",
    "gameTime.gameTimeInSec",
    "squadId",
    "player.id",
    "player.position",
    "actionType",
    "action",
    "result",
    "phase",
    "pressure",
    "start.adjCoordinates.x",
    "start.adjCoordinates.y",
    "end.adjCoordinates.x",
    "end.adjCoordinates.y",
]

event_tables = []

for match_id in season_match_ids:
    path = RAW_DIR / "events" / f"events_{match_id}.json"

    with path.open("r", encoding="utf-8") as file:
        match_events = json.load(file)

    match_table = pd.json_normalize(match_events)

    # Keep a consistent column structure across matches.
    # A field absent from a file becomes missing, not zero.
    match_table = match_table.reindex(columns=event_columns)
    match_table.insert(0, "match_id", match_id)

    event_tables.append(match_table)

season_events = pd.concat(event_tables, ignore_index=True)

# Release the list of individual match tables.
del event_tables, match_table, match_events

expected_events = int(validation_report["event_count"].sum())

print("Expected event rows:", expected_events)
print("Combined event rows:", len(season_events))
print("Matches represented:", season_events["match_id"].nunique())

print(
    "Duplicate match/event IDs:",
    season_events.duplicated(["match_id", "id"]).sum(),
)

print("Missing event IDs:", season_events["id"].isna().sum())

print(
    "Table memory (MB):",
    round(
        season_events.memory_usage(deep=True).sum() / 1024**2,
        1,
    ),
)

display(season_events.head(5))

Expected event rows: 939200
Combined event rows: 939200
Matches represented: 306
Duplicate match/event IDs: 0
Missing event IDs: 0
Table memory (MB): 343.2
   match_id          id  ...  end.adjCoordinates.x  end.adjCoordinates.y
0    122838  4858179098  ...                   NaN                   NaN
1    122838  4858179099  ...                   NaN                   NaN
2    122838  4858179100  ...                   NaN                   NaN
3    122838  4858179101  ...                 -30.9                  -0.1
4    122838  4858179102  ...                  13.4                  27.8

[5 rows x 17 columns]


## Build the season playing-time table

Extract duration and match share for every player-position record. Keep seconds and derive minutes without rounding away precision.

In [7]:
season_duration_rows = []

for match_id in season_match_ids:
    path = (
        RAW_DIR / "player_kpis" / f"player_kpis_{match_id}.json"
    )

    with path.open("r", encoding="utf-8") as file:
        match_kpis = json.load(file)

    for squad_key in ["squadHome", "squadAway"]:
        squad = match_kpis[squad_key]

        for player_record in squad["players"]:
            season_duration_rows.append({
                "match_id": match_id,
                "squadId": squad["id"],
                "player.id": player_record["id"],
                "player.position": player_record["position"],
                "playDuration": player_record["playDuration"],
                "matchShare": player_record["matchShare"],
            })

season_durations = pd.DataFrame(season_duration_rows)
del season_duration_rows

# Convert duration to numeric values and then minutes.
season_durations["playDuration"] = pd.to_numeric(
    season_durations["playDuration"],
    errors="coerce",
)

season_durations["minutes_played"] = (
    season_durations["playDuration"] / 60
)

key_columns = [
    "match_id",
    "squadId",
    "player.id",
    "player.position",
]

print("Player-position records:", len(season_durations))
print("Matches represented:", season_durations["match_id"].nunique())
print("Unique players:", season_durations["player.id"].nunique())

print(
    "Duplicate match/team/player/position records:",
    season_durations.duplicated(key_columns).sum(),
)

print(
    "Rows with missing key fields:",
    season_durations[key_columns].isna().any(axis=1).sum(),
)

print(
    "Missing or invalid durations:",
    season_durations["playDuration"].isna().sum(),
)

print(
    "Negative durations:",
    season_durations["playDuration"].lt(0).sum(),
)

print(
    "Zero-duration records:",
    season_durations["playDuration"].eq(0).sum(),
)

print("\nPOSITIONS AND RECORDED MINUTES")
display(
    season_durations.groupby(
        "player.position",
        dropna=False,
    )
    .agg(
        records=("match_id", "size"),
        total_minutes=("minutes_played", "sum"),
    )
    .round(1)
    .reset_index()
)

Player-position records: 10577
Matches represented: 306
Unique players: 494
Duplicate match/team/player/position records: 0
Rows with missing key fields: 0
Missing or invalid durations: 0
Negative durations: 0
Zero-duration records: 0

POSITIONS AND RECORDED MINUTES
           player.position  records  total_minutes
0       ATTACKING_MIDFIELD     1086        51655.2
1           CENTER_FORWARD     1555        84826.4
2         CENTRAL_DEFENDER     1845       149905.8
3         CENTRAL_MIDFIELD      547        27074.2
4         DEFENSE_MIDFIELD     1608       105874.2
5               GOALKEEPER      614        60699.2
6   LEFT_WINGBACK_DEFENDER      946        60699.2
7              LEFT_WINGER      679        32117.2
8  RIGHT_WINGBACK_DEFENDER      987        60692.9
9             RIGHT_WINGER      710        32119.0


## Select eligible midfielders

Combine all clubs and positions per player. Require at least 900 midfield minutes and a midfield share of at least 50%; the snapshot produces 79 candidates.

In [8]:
midfield_positions = [
    "CENTRAL_MIDFIELD",
    "DEFENSE_MIDFIELD",
    "ATTACKING_MIDFIELD",
]

MIN_MIDFIELD_MINUTES = 900
MIN_MIDFIELD_SHARE = 0.50

# Sum each player's minutes by position across the season.
# A transferred player's minutes are combined across clubs.
player_position_minutes = (
    season_durations.groupby(
        ["player.id", "player.position"]
    )["minutes_played"]
    .sum()
    .unstack("player.position", fill_value=0)
)

player_position_minutes.columns.name = None

# Total minutes include ALL recorded positions.
player_position_minutes["total_minutes"] = (
    player_position_minutes.sum(axis=1)
)

# Ensure the three midfield columns exist.
for position in midfield_positions:
    if position not in player_position_minutes.columns:
        player_position_minutes[position] = 0

player_position_minutes["midfield_minutes"] = (
    player_position_minutes[midfield_positions].sum(axis=1)
)

player_position_minutes["midfield_share"] = (
    player_position_minutes["midfield_minutes"]
    / player_position_minutes["total_minutes"]
)

# Attach player names.
player_season_minutes = (
    player_position_minutes.reset_index()
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="one_to_one",
        indicator="name_match",
    )
)

# Apply our initial eligibility rules.
midfield_pool = player_season_minutes.loc[
    player_season_minutes["midfield_minutes"].ge(
        MIN_MIDFIELD_MINUTES
    )
    & player_season_minutes["midfield_share"].ge(
        MIN_MIDFIELD_SHARE
    )
].copy()

midfield_pool["midfield_share_pct"] = (
    midfield_pool["midfield_share"] * 100
)

print("Players in season:", len(player_season_minutes))
print(
    "Players without a matching name:",
    player_season_minutes["name_match"].eq("left_only").sum(),
)
print("Eligible midfield candidates:", len(midfield_pool))

display(
    midfield_pool[
        [
            "player_name",
            "CENTRAL_MIDFIELD",
            "DEFENSE_MIDFIELD",
            "ATTACKING_MIDFIELD",
            "midfield_minutes",
            "total_minutes",
            "midfield_share_pct",
        ]
    ]
    .sort_values("midfield_minutes", ascending=False)
    .round(1)
    .reset_index(drop=True)
)

Players in season: 494
Players without a matching name: 0
Eligible midfield candidates: 79
             player_name  CENTRAL_MIDFIELD  ...  total_minutes  midfield_share_pct
0   Maximilian Eggestein             161.9  ...         3226.0               100.0
1           Julian Weigl             192.2  ...         3040.5               100.0
2           Granit Xhaka               0.0  ...         3033.4               100.0
3         Angelo Stiller              90.2  ...         3094.2                95.2
4         Atakan Karazor              18.9  ...         2906.0               100.0
..                   ...               ...  ...            ...                 ...
74            Matus Bero             378.1  ...         1799.1                59.9
75      Brenden Aaronson             495.3  ...         1388.8                75.4
76           Adrian Beck             327.0  ...         1073.6                93.0
77            Arne Maier             693.4  ...         1118.7                8

## Identify forward passes

Select candidate midfield PASS events and flag provider SUCCESS actions that move at least 10 metres forward. Missing coordinates cannot qualify.

In [9]:
candidate_ids = midfield_pool["player.id"]

candidate_passes = season_events.loc[
    season_events["player.id"].isin(candidate_ids)
    & season_events["player.position"].isin(midfield_positions)
    & season_events["actionType"].eq("PASS")
].copy()

coordinate_columns = [
    "start.adjCoordinates.x",
    "end.adjCoordinates.x",
]

# Keep missing coordinates as missing.
candidate_passes[coordinate_columns] = (
    candidate_passes[coordinate_columns]
    .apply(pd.to_numeric, errors="coerce")
)

candidate_passes["forward_metres"] = (
    candidate_passes["end.adjCoordinates.x"]
    - candidate_passes["start.adjCoordinates.x"]
)

candidate_passes["has_x_coordinates"] = (
    candidate_passes[coordinate_columns].notna().all(axis=1)
)

FORWARD_THRESHOLD_METRES = 10

candidate_passes["successful_forward_10m"] = (
    candidate_passes["result"].eq("SUCCESS")
    & candidate_passes["has_x_coordinates"]
    & candidate_passes["forward_metres"].ge(
        FORWARD_THRESHOLD_METRES
    )
)

print("Candidate midfield pass events:", len(candidate_passes))
print(
    "Passes missing start or end x:",
    (~candidate_passes["has_x_coordinates"]).sum(),
)

Candidate midfield pass events: 71092
Passes missing start or end x: 991


## Calculate forward-passing rates

Aggregate the flags per player and divide by their midfield minutes. This measures progression frequency, not pass completion.

In [10]:
forward_counts = (
    candidate_passes.groupby("player.id")
    .agg(
        midfield_pass_events=("id", "size"),
        successful_forward_10m_count=(
            "successful_forward_10m", "sum"
        ),
        passes_with_x_coordinates=("has_x_coordinates", "sum"),
    )
    .reset_index()
)

midfield_progression = midfield_pool[
    ["player.id", "player_name", "midfield_minutes"]
].merge(
    forward_counts,
    on="player.id",
    how="left",
    validate="one_to_one",
)

# Report unmatched players rather than assuming their counts are zero.
print(
    "Candidates without pass records:",
    midfield_progression["midfield_pass_events"].isna().sum(),
)

midfield_progression["successful_forward_10m_per90"] = (
    midfield_progression["successful_forward_10m_count"]
    / midfield_progression["midfield_minutes"]
    * 90
)

display(
    midfield_progression.sort_values(
        "successful_forward_10m_per90",
        ascending=False,
    ).head(15).round(2).reset_index(drop=True)
)

Candidates without pass records: 0
    player.id  ... successful_forward_10m_per90
0         281  ...                        15.81
1          98  ...                        13.34
2       27123  ...                        12.98
3         115  ...                        11.56
4       12150  ...                        11.17
5        7873  ...                        10.83
6      103995  ...                        10.79
7        1053  ...                        10.38
8        1481  ...                         9.15
9        1333  ...                         7.83
10       1174  ...                         7.56
11      13493  ...                         7.40
12      23327  ...                         7.35
13      22947  ...                         7.28
14      10460  ...                         7.21

[15 rows x 7 columns]


## Check coordinate coverage

Inspect missing locations by outcome and add pass-volume context before interpreting rates.

In [11]:
# Check whether missing coordinates concentrate in certain outcomes.
coordinate_check = (
    candidate_passes.assign(
        result_label=candidate_passes["result"].fillna("MISSING"),
        missing_coordinates=~candidate_passes["has_x_coordinates"],
    )
    .groupby("result_label")
    .agg(
        pass_events=("id", "size"),
        missing_coordinates=("missing_coordinates", "sum"),
    )
    .reset_index()
)

display(coordinate_check)

# Keep the existing table and add context.
midfield_progression["pass_events_per90"] = (
    midfield_progression["midfield_pass_events"]
    / midfield_progression["midfield_minutes"]
    * 90
)

midfield_progression["coordinate_coverage_pct"] = (
    midfield_progression["passes_with_x_coordinates"]
    / midfield_progression["midfield_pass_events"]
    * 100
)

midfield_progression["successful_forward_10m_per100_passes"] = (
    midfield_progression["successful_forward_10m_count"]
    / midfield_progression["midfield_pass_events"]
    * 100
)

display(
    midfield_progression[
        [
            "player_name",
            "midfield_minutes",
            "pass_events_per90",
            "successful_forward_10m_per90",
            "successful_forward_10m_per100_passes",
            "coordinate_coverage_pct",
        ]
    ]
    .sort_values("successful_forward_10m_per90", ascending=False)
    .head(15)
    .round(2)
    .reset_index(drop=True)
)

  result_label  pass_events  missing_coordinates
0         FAIL        11314                  542
1      MISSING           85                   85
2      NEUTRAL         4517                  364
3      SUCCESS        55176                    0
            player_name  ...  coordinate_coverage_pct
0          Granit Xhaka  ...                    99.48
1        Joshua Kimmich  ...                    98.81
2        Angelo Stiller  ...                    99.02
3              Emre Can  ...                    99.01
4     Exequiel Palacios  ...                    99.15
5        Xaver Schlager  ...                    99.16
6   Aleksandar Pavlovic  ...                    98.95
7         Leon Goretzka  ...                    98.90
8          Kevin Stöger  ...                    97.83
9     Maximilian Arnold  ...                    98.17
10          Kevin Kampl  ...                    99.59
11       Amadou Haidara  ...                    98.77
12       Robert Andrich  ...                    98.82

## Measure forward carrying

Apply the same forward-distance rule to DRIBBLE events. This provider category represents carrying, not only beating an opponent.

In [12]:
candidate_carries = season_events.loc[
    season_events["player.id"].isin(midfield_pool["player.id"])
    & season_events["player.position"].isin(midfield_positions)
    & season_events["actionType"].eq("DRIBBLE")
].copy()

coordinate_columns = [
    "start.adjCoordinates.x",
    "end.adjCoordinates.x",
]

candidate_carries[coordinate_columns] = (
    candidate_carries[coordinate_columns]
    .apply(pd.to_numeric, errors="coerce")
)

candidate_carries["has_x_coordinates"] = (
    candidate_carries[coordinate_columns].notna().all(axis=1)
)

candidate_carries["forward_metres"] = (
    candidate_carries["end.adjCoordinates.x"]
    - candidate_carries["start.adjCoordinates.x"]
)

candidate_carries["successful_forward_carry_10m"] = (
    candidate_carries["result"].eq("SUCCESS")
    & candidate_carries["has_x_coordinates"]
    & candidate_carries["forward_metres"].ge(
        FORWARD_THRESHOLD_METRES
    )
)

# Inspect outcomes and coordinate coverage.
carry_coordinate_check = (
    candidate_carries.assign(
        result_label=candidate_carries["result"].fillna("MISSING"),
        missing_coordinates=~candidate_carries["has_x_coordinates"],
    )
    .groupby("result_label")
    .agg(
        carry_events=("id", "size"),
        missing_coordinates=("missing_coordinates", "sum"),
    )
    .reset_index()
)

display(carry_coordinate_check)

# Aggregate to one row per player.
carry_counts = (
    candidate_carries.groupby("player.id")
    .agg(
        carry_events=("id", "size"),
        successful_forward_carry_10m_count=(
            "successful_forward_carry_10m", "sum"
        ),
    )
    .reset_index()
)

# Combine passing and carrying metrics.
midfield_passing_carrying = midfield_progression.merge(
    carry_counts,
    on="player.id",
    how="left",
    validate="one_to_one",
)

print(
    "Candidates without carry records:",
    midfield_passing_carrying["carry_events"].isna().sum(),
)

midfield_passing_carrying["successful_forward_carry_10m_per90"] = (
    midfield_passing_carrying["successful_forward_carry_10m_count"]
    / midfield_passing_carrying["midfield_minutes"]
    * 90
)

display(
    midfield_passing_carrying[
        [
            "player_name",
            "midfield_minutes",
            "carry_events",
            "successful_forward_carry_10m_count",
            "successful_forward_carry_10m_per90",
            "successful_forward_10m_per90",
        ]
    ]
    .sort_values(
        "successful_forward_carry_10m_per90",
        ascending=False,
    )
    .head(15)
    .round(2)
    .reset_index(drop=True)
)

  result_label  carry_events  missing_coordinates
0         FAIL          2240                   36
1      NEUTRAL          1346                 1343
2      SUCCESS         43778                    0
Candidates without carry records: 0
             player_name  ...  successful_forward_10m_per90
0          Florian Wirtz  ...                          6.99
1         Xaver Schlager  ...                         10.83
2           Brajan Gruda  ...                          3.06
3           Granit Xhaka  ...                         15.81
4          Leon Goretzka  ...                         10.38
5          Jamal Musiala  ...                          4.02
6          Romano Schmid  ...                          5.15
7   Leonardo Bittencourt  ...                          5.57
8           Kevin Stöger  ...                          9.15
9              Manu Koné  ...                          4.51
10     Denis Huseinbasic  ...                          5.97
11       Andrej Kramaric  ...               

## Measure final-third entries

Count successful moves crossing x=17.5. Final-third entry does not additionally require a 10-metre distance.

In [13]:
FINAL_THIRD_X = 17.5

# Combine the candidate midfield passes and carries already selected.
candidate_moves = pd.concat(
    [candidate_passes, candidate_carries],
    ignore_index=True,
)

crosses_final_third = (
    candidate_moves["result"].eq("SUCCESS")
    & candidate_moves["has_x_coordinates"]
    & candidate_moves["start.adjCoordinates.x"].le(FINAL_THIRD_X)
    & candidate_moves["end.adjCoordinates.x"].gt(FINAL_THIRD_X)
)

candidate_moves["final_third_pass_entry"] = (
    crosses_final_third
    & candidate_moves["actionType"].eq("PASS")
)

candidate_moves["final_third_carry_entry"] = (
    crosses_final_third
    & candidate_moves["actionType"].eq("DRIBBLE")
)

entry_counts = (
    candidate_moves.groupby("player.id")
    .agg(
        final_third_pass_entries=("final_third_pass_entry", "sum"),
        final_third_carry_entries=("final_third_carry_entry", "sum"),
    )
    .reset_index()
)

midfield_territory = midfield_passing_carrying.merge(
    entry_counts,
    on="player.id",
    how="left",
    validate="one_to_one",
)

print(
    "Candidates without entry-summary records:",
    midfield_territory["final_third_pass_entries"].isna().sum(),
)

for entry_type in ["pass", "carry"]:
    count_column = f"final_third_{entry_type}_entries"

    midfield_territory[f"{count_column}_per90"] = (
        midfield_territory[count_column]
        / midfield_territory["midfield_minutes"]
        * 90
    )

display(
    midfield_territory[
        [
            "player_name",
            "midfield_minutes",
            "final_third_pass_entries_per90",
            "final_third_carry_entries_per90",
        ]
    ]
    .sort_values("final_third_pass_entries_per90", ascending=False)
    .head(15)
    .round(2)
    .reset_index(drop=True)
)

Candidates without entry-summary records: 0
            player_name  ...  final_third_carry_entries_per90
0          Granit Xhaka  ...                             2.14
1        Joshua Kimmich  ...                             1.53
2   Aleksandar Pavlovic  ...                             1.50
3     Exequiel Palacios  ...                             1.43
4        Angelo Stiller  ...                             1.37
5        Xaver Schlager  ...                             1.84
6         Leon Goretzka  ...                             2.47
7        Robert Andrich  ...                             1.47
8          Kevin Stöger  ...                             2.35
9              Emre Can  ...                             1.15
10       Amadou Haidara  ...                             1.16
11    Maximilian Arnold  ...                             0.61
12      Florian Neuhaus  ...                             1.48
13       Atakan Karazor  ...                             1.21
14        Florian Wirtz  .

## Read the KPI dictionary

Inspect provider definitions before choosing creation metrics. Definitions are cached locally.

In [14]:
kpi_definitions_data = metadata("kpi_definitions.json")
print("KPI definitions:", len(kpi_definitions_data))


KPI definitions: 103


## Find creation metrics

Search dictionary text rather than guessing KPI names or IDs.

In [15]:
kpi_definitions = pd.json_normalize(kpi_definitions_data)

print("Dictionary columns:", kpi_definitions.columns.tolist())

# Search all fields so we don't assume a particular name column.
definition_text = (
    kpi_definitions.fillna("")
    .astype(str)
    .agg(" ".join, axis=1)
)

creation_mask = definition_text.str.contains(
    r"assist|key.?pass|chance|expected.?goal|expected.?assist",
    case=False,
    regex=True,
    na=False,
)

creation_definitions = kpi_definitions.loc[creation_mask]

print("Matching definitions:", len(creation_definitions))

print(
    creation_definitions.head(25).to_string(
        index=False,
        max_colwidth=160,
    )
)

Dictionary columns: ['id', 'name', 'parentKpi', 'context', 'details.label', 'details.definition', 'details.meaning', 'details', 'parentKpi.id', 'parentKpi.label', 'parentKpi.name', 'parentKpi.definition', 'parentKpi.meaning', 'context.label', 'context.name', 'context.definition', 'context.meaning']
Matching definitions: 11
  id                                name  parentKpi  context                       details.label                                                                                                                                               details.definition                                                                                                                                                  details.meaning  details  parentKpi.id parentKpi.label parentKpi.name parentKpi.definition parentKpi.meaning context.label context.name context.definition context.meaning
  20           BALL_LOSS_ADDED_OPPONENTS        NaN      NaN           Ball Loss Added Opponents Sum 

## Inspect creation KPI availability

Check three named KPI definitions and their presence in a sample match. Absence is not assumed to mean zero.

In [16]:
creation_kpi_ids = [1780, 1693, 1782]

# Print full definitions without table truncation.
for definition in kpi_definitions_data:
    if definition["id"] in creation_kpi_ids:
        print(f"\nID: {definition['id']}")
        print("NAME:", definition["name"])
        print(
            "DEFINITION:",
            definition.get("details", {}).get("definition", ""),
        )

# Inspect availability in our example match.
path = (
    RAW_DIR / "player_kpis"
    / f"player_kpis_{example_match_id}.json"
)

with path.open("r", encoding="utf-8") as file:
    example_kpis = json.load(file)

availability_rows = []

for squad_key in ["squadHome", "squadAway"]:
    for player_record in example_kpis[squad_key]["players"]:
        values = {
            item["kpiId"]: item["value"]
            for item in player_record["kpis"]
        }

        row = {
            "player.id": player_record["id"],
            "position": player_record["position"],
        }

        for kpi_id in creation_kpi_ids:
            # Missing stays missing; it must not become zero.
            row[f"kpi_{kpi_id}"] = values.get(kpi_id)

        availability_rows.append(row)

creation_sample = pd.DataFrame(availability_rows)

print("\nKPI AVAILABILITY IN EXAMPLE MATCH")
for kpi_id in creation_kpi_ids:
    column = f"kpi_{kpi_id}"
    present = creation_sample[column].notna().sum()
    print(f"{kpi_id}: {present}/{len(creation_sample)} records populated")

display(creation_sample.head(5))


ID: 1693
NAME: SHOT_XG_FROM_PASSES
DEFINITION: Shot xG from Passes measures the total expected goals (xG) generated from shots that directly followed a player’s passes. The model does not estimate probabilities itself — it simply assigns the xG value of each shot to the pass that created it. All passes that directly lead to a shot are included; passes that don’t result in a shot are ignored.

ID: 1780
NAME: SHOT_ASSISTS
DEFINITION: A shot assist is awarded when a player delivers the final pass or set piece that directly leads to a teammate taking a shot. This includes all types of deliveries such as short or long passes, crosses, through balls, corners, and free kicks, provided the action immediately precedes a shot attempt. The metric is a simple count of such events, independent of whether the resulting shot was on target or resulted in a goal. Only passes to the final third or opponent box are awarded.

ID: 1782
NAME: EXPECTED_GOAL_ASSISTS
DEFINITION: A model-based estimate of how 

## Measure creation record coverage

Extract candidate midfield KPIs and reconcile their four-part keys to the playing-time table.

In [17]:
creation_metrics = {
    1780: "shot_assists",
    1693: "shot_xg_from_passes",
    1782: "expected_goal_assists",
}

candidate_ids = set(midfield_pool["player.id"])
creation_rows = []

for match_id in season_match_ids:
    path = (
        RAW_DIR / "player_kpis" / f"player_kpis_{match_id}.json"
    )

    with path.open("r", encoding="utf-8") as file:
        match_kpis = json.load(file)

    for squad_key in ["squadHome", "squadAway"]:
        squad = match_kpis[squad_key]

        for player in squad["players"]:
            if (
                player["id"] not in candidate_ids
                or player["position"] not in midfield_positions
            ):
                continue

            values = {
                item["kpiId"]: item["value"]
                for item in player["kpis"]
            }

            row = {
                "match_id": match_id,
                "squadId": squad["id"],
                "player.id": player["id"],
                "player.position": player["position"],
            }

            for kpi_id, metric_name in creation_metrics.items():
                row[metric_name] = values.get(kpi_id)

            creation_rows.append(row)

season_creation_records = pd.DataFrame(creation_rows)

# Reconcile against the candidate midfield duration records.
expected_keys = season_durations.loc[
    season_durations["player.id"].isin(candidate_ids)
    & season_durations["player.position"].isin(midfield_positions),
    key_columns,
]

creation_key_check = expected_keys.merge(
    season_creation_records[key_columns],
    on=key_columns,
    how="outer",
    validate="one_to_one",
    indicator=True,
)

if not creation_key_check["_merge"].eq("both").all():
    raise ValueError("Creation records do not match duration records.")

print("Candidate midfield records:", len(season_creation_records))
print("Record keys match playing-time table: YES")

coverage = pd.DataFrame({
    "populated_records": season_creation_records[
        list(creation_metrics.values())
    ].notna().sum(),
    "total_records": len(season_creation_records),
})

coverage["coverage_pct"] = (
    coverage["populated_records"] / coverage["total_records"] * 100
)

display(coverage.round(2))

Candidate midfield records: 2115
Record keys match playing-time table: YES
                       populated_records  total_records  coverage_pct
shot_assists                         936           2115         44.26
shot_xg_from_passes                  996           2115         47.09
expected_goal_assists               2015           2115         95.27


## Avoid incomplete season totals

Only present a full-season expected-assist rate when every relevant record has a value. Partial data remains available for later coverage-aware exploration.

In [18]:
creation_totals = (
    season_creation_records.groupby("player.id")
    ["expected_goal_assists"]
    .agg(
        position_records="size",
        populated_records="count",
        expected_goal_assists_total="sum",
    )
    .reset_index()
)

complete = (
    creation_totals["populated_records"]
    == creation_totals["position_records"]
)

# Prevent partial totals from appearing as full-season values.
creation_totals["expected_goal_assists_total"] = (
    creation_totals["expected_goal_assists_total"].where(complete)
)

midfield_with_creation = midfield_territory.merge(
    creation_totals,
    on="player.id",
    how="left",
    validate="one_to_one",
)

midfield_with_creation["expected_goal_assists_per90"] = (
    midfield_with_creation["expected_goal_assists_total"]
    / midfield_with_creation["midfield_minutes"]
    * 90
)

print(
    "Candidates with complete expected-assist coverage:",
    midfield_with_creation["expected_goal_assists_total"]
    .notna().sum(),
    "/",
    len(midfield_with_creation),
)

display(
    midfield_with_creation.loc[
        midfield_with_creation["expected_goal_assists_per90"].notna(),
        [
            "player_name",
            "midfield_minutes",
            "expected_goal_assists_total",
            "expected_goal_assists_per90",
        ],
    ]
    .sort_values("expected_goal_assists_per90", ascending=False)
    .head(15)
    .round(3)
    .reset_index(drop=True)
)

Candidates with complete expected-assist coverage: 27 / 79
             player_name  ...  expected_goal_assists_per90
0          Thomas Müller  ...                        0.367
1          Jonas Hofmann  ...                        0.361
2           Farès Chaïbi  ...                        0.350
3          Florian Wirtz  ...                        0.321
4          Romano Schmid  ...                        0.233
5           Brajan Gruda  ...                        0.233
6             Marco Reus  ...                        0.213
7            Enzo Millot  ...                        0.207
8   Leonardo Bittencourt  ...                        0.167
9       Yannick Gerhardt  ...                        0.154
10         Florian Kainz  ...                        0.150
11         Leon Goretzka  ...                        0.144
12     Exequiel Palacios  ...                        0.143
13          Rubén Vargas  ...                        0.140
14       Marcel Sabitzer  ...                        0.1

## Measure coverage by minutes

A missing short appearance and a missing full match are different exposure losses; report both record and minutes coverage.

In [19]:
creation_coverage_records = season_creation_records.merge(
    season_durations[key_columns + ["minutes_played"]],
    on=key_columns,
    how="left",
    validate="one_to_one",
)

creation_coverage_records["has_expected_assists"] = (
    creation_coverage_records["expected_goal_assists"].notna()
)

creation_coverage_records["covered_minutes"] = (
    creation_coverage_records["minutes_played"].where(
        creation_coverage_records["has_expected_assists"],
        0,
    )
)

player_creation_coverage = (
    creation_coverage_records.groupby("player.id")
    .agg(
        total_records=("match_id", "size"),
        populated_records=("has_expected_assists", "sum"),
        midfield_minutes=("minutes_played", "sum"),
        covered_minutes=("covered_minutes", "sum"),
    )
    .reset_index()
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="one_to_one",
    )
)

player_creation_coverage["missing_records"] = (
    player_creation_coverage["total_records"]
    - player_creation_coverage["populated_records"]
)

player_creation_coverage["minutes_coverage_pct"] = (
    player_creation_coverage["covered_minutes"]
    / player_creation_coverage["midfield_minutes"]
    * 100
)

display(
    player_creation_coverage[
        [
            "player_name",
            "missing_records",
            "midfield_minutes",
            "covered_minutes",
            "minutes_coverage_pct",
        ]
    ]
    .sort_values("minutes_coverage_pct")
    .head(15)
    .round(2)
    .reset_index(drop=True)
)

         player_name  missing_records  ...  covered_minutes  minutes_coverage_pct
0       Rani Khedira                4  ...          1311.72                 88.26
1        Kevin Kampl                6  ...          1355.70                 89.62
2     Bartol Franjic                5  ...          1133.24                 90.15
3        Merlin Röhl                3  ...          1043.37                 93.51
4          Alex Král                4  ...          1376.81                 95.68
5     Robert Andrich                6  ...          1407.15                 95.75
6    Elvis Rexhbecaj                1  ...          2086.53                 95.94
7     Xaver Schlager                1  ...          2590.71                 96.24
8     Atakan Karazor                2  ...          2798.39                 96.30
9       Jae-sung Lee                1  ...          1829.46                 96.43
10  Brenden Aaronson                2  ...          1013.65                 96.76
11    Aïssa Laïd

## Trace a missing value to source

Inspect one missing KPI in the original JSON instead of automatically filling it with zero.

In [20]:
missing_example = creation_coverage_records.loc[
    ~creation_coverage_records["has_expected_assists"]
].iloc[0]

match_id = int(missing_example["match_id"])

path = RAW_DIR / "player_kpis" / f"player_kpis_{match_id}.json"

with path.open("r", encoding="utf-8") as file:
    original = json.load(file)

for squad_key in ["squadHome", "squadAway"]:
    squad = original[squad_key]

    if squad["id"] != missing_example["squadId"]:
        continue

    for player in squad["players"]:
        if (
            player["id"] == missing_example["player.id"]
            and player["position"] == missing_example["player.position"]
        ):
            matching_entries = [
                item for item in player["kpis"]
                if item["kpiId"] == 1782
            ]

            print("Match:", match_id)
            print("Player ID:", player["id"])
            print("Position:", player["position"])
            print("Minutes:", round(player["playDuration"] / 60, 2))
            print("Original KPI 1782 entries:", matching_entries)

Match: 122840
Player ID: 7369
Position: DEFENSE_MIDFIELD
Minutes: 21.18
Original KPI 1782 entries: []


## Measure selected defensive activity

Count interceptions and loose-ball regains, then calculate per-90 rates. These two actions are not a complete defensive-quality measure.

In [21]:
# All events recorded for our candidates in midfield positions.
candidate_midfield_events = season_events.loc[
    season_events["player.id"].isin(midfield_pool["player.id"])
    & season_events["player.position"].isin(midfield_positions)
].copy()

candidate_midfield_events["is_interception"] = (
    candidate_midfield_events["actionType"].eq("INTERCEPTION")
)

candidate_midfield_events["is_loose_ball_regain"] = (
    candidate_midfield_events["actionType"].eq("LOOSE_BALL_REGAIN")
)

# Keep all recorded candidate events in the grouping.
# A player with no interceptions therefore receives a count of zero.
defensive_counts = (
    candidate_midfield_events.groupby("player.id")
    .agg(
        recorded_midfield_events=("id", "size"),
        interceptions=("is_interception", "sum"),
        loose_ball_regains=("is_loose_ball_regain", "sum"),
    )
    .reset_index()
)

midfield_profile = midfield_territory.merge(
    defensive_counts,
    on="player.id",
    how="left",
    validate="one_to_one",
)

print(
    "Candidates without midfield event records:",
    midfield_profile["recorded_midfield_events"].isna().sum(),
)

for metric in ["interceptions", "loose_ball_regains"]:
    midfield_profile[f"{metric}_per90"] = (
        midfield_profile[metric]
        / midfield_profile["midfield_minutes"]
        * 90
    )

display(
    midfield_profile[
        [
            "player_name",
            "midfield_minutes",
            "interceptions_per90",
            "loose_ball_regains_per90",
            "successful_forward_10m_per90",
            "successful_forward_carry_10m_per90",
        ]
    ]
    .sort_values("interceptions_per90", ascending=False)
    .head(15)
    .round(2)
    .reset_index(drop=True)
)

Candidates without midfield event records: 0
            player_name  ...  successful_forward_carry_10m_per90
0             Tom Krauß  ...                                1.50
1           Senne Lynen  ...                                0.69
2           Anton Stach  ...                                3.11
3         Ellyes Skhiri  ...                                0.98
4        Bartol Franjic  ...                                1.72
5     Patrick Osterhage  ...                                2.25
6         Jan Schöppner  ...                                1.09
7        Amadou Haidara  ...                                1.55
8       Kristijan Jakic  ...                                1.12
9     Exequiel Palacios  ...                                2.31
10      Lennard Maloney  ...                                0.53
11          Eric Martel  ...                                1.46
12  Aleksandar Pavlovic  ...                                2.46
13       Robert Andrich  ...                 

## Convert rates into percentiles

Use average ranks for ties within the fixed 79-player pool. A passing percentile of 100 means the highest progression rate in this pool.

In [22]:
profile_metrics = {
    "successful_forward_10m_per90": "forward_pass",
    "successful_forward_carry_10m_per90": "forward_carry",
    "final_third_pass_entries_per90": "third_entry_pass",
    "final_third_carry_entries_per90": "third_entry_carry",
    "interceptions_per90": "interception",
    "loose_ball_regains_per90": "loose_regain",
}

# Keep the existing counts and rates.
midfield_comparison = midfield_profile.copy()

# Check completeness before comparing ranks.
print("Players:", len(midfield_comparison))
print(
    "Missing metric values:",
    midfield_comparison[list(profile_metrics)].isna().sum().sum(),
)

for metric, short_name in profile_metrics.items():
    midfield_comparison[f"{short_name}_pctile"] = (
        midfield_comparison[metric]
        .rank(method="average", pct=True)
        * 100
    )

# Attach recorded position minutes to help interpret roles.
position_minutes = midfield_pool[
    ["player.id"] + midfield_positions
]

midfield_comparison = midfield_comparison.merge(
    position_minutes,
    on="player.id",
    how="left",
    validate="one_to_one",
)

# Share of MIDFIELD minutes spent in each recorded position.
for position in midfield_positions:
    midfield_comparison[f"{position}_share_pct"] = (
        midfield_comparison[position]
        / midfield_comparison["midfield_minutes"]
        * 100
    )

# Compare a few different profiles, without making a shortlist yet.
example_names = [
    "Anton Stach",
    "Exequiel Palacios",
    "Angelo Stiller",
    "Leon Goretzka",
    "Florian Wirtz",
    "Ellyes Skhiri",
]

percentile_columns = [
    f"{short_name}_pctile"
    for short_name in profile_metrics.values()
]

display(
    midfield_comparison.loc[
        midfield_comparison["player_name"].isin(example_names),
        ["player_name", "midfield_minutes"] + percentile_columns,
    ]
    .sort_values("player_name")
    .round(1)
    .reset_index(drop=True)
)

print("\nRECORDED POSITION SHARES — SAME PLAYERS")

display(
    midfield_comparison.loc[
        midfield_comparison["player_name"].isin(example_names),
        ["player_name"] + [
            f"{position}_share_pct"
            for position in midfield_positions
        ],
    ]
    .sort_values("player_name")
    .round(1)
    .reset_index(drop=True)
)

Players: 79
Missing metric values: 0
         player_name  ...  loose_regain_pctile
0     Angelo Stiller  ...                 53.2
1        Anton Stach  ...                 98.7
2      Ellyes Skhiri  ...                 69.6
3  Exequiel Palacios  ...                 46.8
4      Florian Wirtz  ...                 16.5
5      Leon Goretzka  ...                 45.6

[6 rows x 8 columns]

RECORDED POSITION SHARES — SAME PLAYERS
         player_name  ...  ATTACKING_MIDFIELD_share_pct
0     Angelo Stiller  ...                           2.9
1        Anton Stach  ...                           3.7
2      Ellyes Skhiri  ...                           0.0
3  Exequiel Palacios  ...                           0.3
4      Florian Wirtz  ...                         100.0
5      Leon Goretzka  ...                           0.0

[6 rows x 4 columns]


## Build three scoring scenarios

Combine passing, carrying and defensive activity under three weight sets. Rank rounded scores to six decimals consistently with SQL; raw scores remain unrounded.

In [23]:
midfield_screening = midfield_comparison.copy()

midfield_screening["passing_component"] = (
    midfield_screening["forward_pass_pctile"]
)

midfield_screening["carrying_component"] = (
    midfield_screening["forward_carry_pctile"]
)

midfield_screening["defensive_activity_component"] = (
    midfield_screening[
        ["interception_pctile", "loose_regain_pctile"]
    ].mean(axis=1)
)

# Alternative briefs test whether our results depend on the weights.
weight_scenarios = {
    "balanced": (0.35, 0.35, 0.30),
    "passing_focus": (0.50, 0.20, 0.30),
    "carrying_focus": (0.20, 0.50, 0.30),
}

for scenario, weights in weight_scenarios.items():
    passing_weight, carrying_weight, defensive_weight = weights

    midfield_screening[f"{scenario}_score"] = (
        midfield_screening["passing_component"] * passing_weight
        + midfield_screening["carrying_component"] * carrying_weight
        + midfield_screening["defensive_activity_component"]
        * defensive_weight
    )

    midfield_screening[f"{scenario}_rank"] = (
        midfield_screening[f"{scenario}_score"]
        .round(6)
        .rank(ascending=False, method="min")
        .astype(int)
    )

rank_columns = [
    f"{scenario}_rank"
    for scenario in weight_scenarios
]

midfield_screening["best_scenario_rank"] = (
    midfield_screening[rank_columns].min(axis=1)
)

midfield_screening["worst_scenario_rank"] = (
    midfield_screening[rank_columns].max(axis=1)
)

display(
    midfield_screening[
        [
            "player_name",
            "midfield_minutes",
            "passing_component",
            "carrying_component",
            "defensive_activity_component",
            "balanced_score",
            "balanced_rank",
            "passing_focus_rank",
            "carrying_focus_rank",
            "best_scenario_rank",
            "worst_scenario_rank",
        ]
    ]
    .sort_values(
        ["balanced_score", "player_name"],
        ascending=[False, True],
    )
    .head(15)
    .round(1)
    .reset_index(drop=True)
)

            player_name  ...  worst_scenario_rank
0        Xaver Schlager  ...                    1
1          Granit Xhaka  ...                    2
2           Anton Stach  ...                    4
3              Emre Can  ...                    6
4          Kevin Stöger  ...                    5
5         Leon Goretzka  ...                    6
6           Kevin Kampl  ...                    9
7        Joshua Kimmich  ...                    8
8     Exequiel Palacios  ...                   12
9   Aleksandar Pavlovic  ...                   11
10    Denis Huseinbasic  ...                   16
11       Angelo Stiller  ...                   18
12       Amadou Haidara  ...                   19
13        Florian Wirtz  ...                   20
14       Robert Andrich  ...                   21

[15 rows x 11 columns]


## Add club and role context

Combine transferred players across clubs and show position-minute shares to make profiles interpretable.

In [24]:
# Find the clubs represented in each candidate's midfield records.
candidate_club_records = season_durations.loc[
    season_durations["player.id"].isin(midfield_pool["player.id"])
    & season_durations["player.position"].isin(midfield_positions),
    ["player.id", "squadId"],
].drop_duplicates()

candidate_club_records = candidate_club_records.merge(
    squad_lookup,
    on="squadId",
    how="left",
    validate="many_to_one",
)

print(
    "Unmatched club records:",
    candidate_club_records["team_name"].isna().sum(),
)

clubs_by_player = (
    candidate_club_records.groupby("player.id")["team_name"]
    .agg(lambda names: " / ".join(sorted(names.dropna().unique())))
    .reset_index(name="clubs_2023_24")
)

# midfield_screening already contains the recorded position shares.
screening_review = midfield_screening.merge(
    clubs_by_player,
    on="player.id",
    how="left",
    validate="one_to_one",
)

review_columns = [
    "player_name",
    "clubs_2023_24",
    "midfield_minutes",
    "balanced_rank",
    "best_scenario_rank",
    "worst_scenario_rank",
    "CENTRAL_MIDFIELD_share_pct",
    "DEFENSE_MIDFIELD_share_pct",
    "ATTACKING_MIDFIELD_share_pct",
]

display(
    screening_review[review_columns]
    .sort_values(["balanced_rank", "player_name"])
    .head(15)
    .round(1)
    .reset_index(drop=True)
)

Unmatched club records: 0
            player_name  ... ATTACKING_MIDFIELD_share_pct
0        Xaver Schlager  ...                          0.0
1          Granit Xhaka  ...                          0.0
2           Anton Stach  ...                          3.7
3              Emre Can  ...                          0.0
4          Kevin Stöger  ...                         36.8
5         Leon Goretzka  ...                          0.0
6           Kevin Kampl  ...                          7.1
7        Joshua Kimmich  ...                          0.0
8     Exequiel Palacios  ...                          0.3
9   Aleksandar Pavlovic  ...                          0.0
10    Denis Huseinbasic  ...                          2.5
11       Angelo Stiller  ...                          2.9
12       Amadou Haidara  ...                          0.0
13        Florian Wirtz  ...                        100.0
14       Robert Andrich  ...                          1.1

[15 rows x 9 columns]


## Define the two season halves

Provider matchdays are zero-based. Matchdays 0–16 become H1 and 17–33 become H2.

In [25]:
match_halves = available_matches[
    ["id", "matchDay.index"]
].rename(columns={"id": "match_id"}).copy()

# Confirm the expected zero-based matchday numbering.
if set(match_halves["matchDay.index"]) != set(range(34)):
    raise ValueError("Unexpected matchday numbering; inspect before splitting.")

match_halves["season_half"] = match_halves["matchDay.index"].apply(
    lambda matchday: "H1" if matchday < 17 else "H2"
)

print("Matches per half:")
print(match_halves["season_half"].value_counts().to_string())

Matches per half:
season_half
H1    153
H2    153


## Compare half-season exposure

Aggregate minutes and recorded progression per half. This is participation/sample context, not a performance-consistency score.

In [26]:
match_player_keys = ["match_id", "player.id"]

# Playing time: combine midfield positions within each match.
minutes_by_match = (
    season_durations.loc[
        season_durations["player.id"].isin(midfield_pool["player.id"])
        & season_durations["player.position"].isin(midfield_positions)
    ]
    .groupby(match_player_keys)["minutes_played"]
    .sum()
    .reset_index()
)

# Count the same qualifying actions used in our season metrics.
passes_by_match = (
    candidate_passes.groupby(match_player_keys)
    ["successful_forward_10m"]
    .sum()
    .reset_index(name="forward_passes")
)

carries_by_match = (
    candidate_carries.groupby(match_player_keys)
    ["successful_forward_carry_10m"]
    .sum()
    .reset_index(name="forward_carries")
)

half_match_records = (
    minutes_by_match
    .merge(
        passes_by_match,
        on=match_player_keys,
        how="left",
        validate="one_to_one",
    )
    .merge(
        carries_by_match,
        on=match_player_keys,
        how="left",
        validate="one_to_one",
    )
    .merge(
        match_halves[["match_id", "season_half"]],
        on="match_id",
        how="left",
        validate="many_to_one",
    )
)

# No matching action rows means zero RECORDED actions of that type.
count_columns = ["forward_passes", "forward_carries"]
half_match_records[count_columns] = (
    half_match_records[count_columns].fillna(0)
)

half_profiles = (
    half_match_records.groupby(["player.id", "season_half"])
    .agg(
        midfield_minutes=("minutes_played", "sum"),
        forward_passes=("forward_passes", "sum"),
        forward_carries=("forward_carries", "sum"),
    )
    .reset_index()
    .merge(
        player_lookup,
        on="player.id",
        how="left",
        validate="many_to_one",
    )
)

half_profiles["forward_passes_per90"] = (
    half_profiles["forward_passes"]
    / half_profiles["midfield_minutes"] * 90
)

half_profiles["forward_carries_per90"] = (
    half_profiles["forward_carries"]
    / half_profiles["midfield_minutes"] * 90
)

half_profiles["sample_flag"] = half_profiles["midfield_minutes"].apply(
    lambda minutes: "450+ minutes" if minutes >= 450 else "SMALL SAMPLE"
)

# Inspect the current top ten from the full-season screening.
top_ten_ids = (
    midfield_screening.sort_values("balanced_rank")
    .head(10)["player.id"]
)

display(
    half_profiles.loc[
        half_profiles["player.id"].isin(top_ten_ids),
        [
            "player_name",
            "season_half",
            "midfield_minutes",
            "forward_passes_per90",
            "forward_carries_per90",
            "sample_flag",
        ],
    ]
    .sort_values(["player_name", "season_half"])
    .round(2)
    .reset_index(drop=True)
)

            player_name season_half  ...  forward_carries_per90   sample_flag
0   Aleksandar Pavlovic          H1  ...                   3.64  SMALL SAMPLE
1   Aleksandar Pavlovic          H2  ...                   2.04  450+ minutes
2           Anton Stach          H1  ...                   3.38  450+ minutes
3           Anton Stach          H2  ...                   2.82  450+ minutes
4              Emre Can          H1  ...                   3.02  450+ minutes
5              Emre Can          H2  ...                   2.72  450+ minutes
6     Exequiel Palacios          H1  ...                   2.14  450+ minutes
7     Exequiel Palacios          H2  ...                   2.70  450+ minutes
8          Granit Xhaka          H1  ...                   4.72  450+ minutes
9          Granit Xhaka          H2  ...                   3.95  450+ minutes
10       Joshua Kimmich          H1  ...                   3.05  450+ minutes
11       Joshua Kimmich          H2  ...                   2.22 

## Build the initial review list

Require at least 450 midfield minutes in each half. Of 79 candidates, 58 meet the rule; the first five balanced ranks form the historical shortlist.

In [27]:
# One row per player, with midfield minutes in each season half.
half_minutes = (
    half_profiles.pivot(
        index="player.id",
        columns="season_half",
        values="midfield_minutes",
    )
    .reindex(columns=["H1", "H2"])
    .fillna(0)
    .rename(columns={
        "H1": "h1_midfield_minutes",
        "H2": "h2_midfield_minutes",
    })
    .reset_index()
)

half_minutes.columns.name = None

screening_with_samples = screening_review.merge(
    half_minutes,
    on="player.id",
    how="left",
    validate="one_to_one",
)

screening_with_samples["sufficient_minutes_both_halves"] = (
    screening_with_samples["h1_midfield_minutes"].ge(450)
    & screening_with_samples["h2_midfield_minutes"].ge(450)
)

initial_review_list = (
    screening_with_samples.loc[
        screening_with_samples["sufficient_minutes_both_halves"]
    ]
    .sort_values(["balanced_rank", "player_name"])
    .head(5)
    .copy()
)

display(
    initial_review_list[
        [
            "player_name",
            "clubs_2023_24",
            "balanced_rank",
            "best_scenario_rank",
            "worst_scenario_rank",
            "h1_midfield_minutes",
            "h2_midfield_minutes",
            "successful_forward_10m_per90",
            "successful_forward_carry_10m_per90",
            "interceptions_per90",
            "loose_ball_regains_per90",
        ]
    ]
    .round(2)
    .reset_index(drop=True)
)

      player_name  ... loose_ball_regains_per90
0  Xaver Schlager  ...                     7.32
1    Granit Xhaka  ...                     7.60
2     Anton Stach  ...                     9.97
3        Emre Can  ...                     7.63
4    Kevin Stöger  ...                     7.10

[5 rows x 11 columns]


## Save outputs

Save named intermediate tables for the next notebook. Generated files stay local and are excluded from Git.

In [28]:
# Fixed-snapshot acceptance checks.
assert len(season_events) == 939200
assert len(season_durations) == 10577
assert len(midfield_pool) == 79
assert int(screening_with_samples["sufficient_minutes_both_halves"].sum()) == 58
assert midfield_screening.loc[midfield_screening["player_name"].isin(["Xaver Schlager", "Granit Xhaka"]), "passing_focus_rank"].eq(1).all()
assert initial_review_list["player_name"].tolist() == ["Xaver Schlager", "Granit Xhaka", "Anton Stach", "Emre Can", "Kevin Stöger"]
save_context("core_analysis", globals(), BASE_VARIABLES)
REPORTS_DIR.mkdir(exist_ok=True)
validation_report.to_csv(REPORTS_DIR / "raw_file_validation.csv", index=False)
screening_with_samples.to_csv(REPORTS_DIR / "candidate_screening.csv", index=False)
initial_review_list.to_csv(REPORTS_DIR / "initial_shortlist.csv", index=False)
print("Core snapshot checks passed.")


Saved local checkpoint: core_analysis.pkl
Core snapshot checks passed.
